# Bruun Rule Validation (Pre-2000 Baseline, 2000->2024 Pseudo-Forecast)

This notebook evaluates which Bruun-rule equation best reproduces NZCCD shoreline change using a pseudo-forecast setup.

Assumption used here:
- Pre-2000 shoreline conditions are treated as the reference baseline.
- Historic trend is calibrated from first observed shoreline to the shoreline position closest to/at 2000.
- Projection period is from that pre-2000 baseline to latest observed shoreline up to 2024.
- SLR forcing uses 2020 medium-confidence NZSeaRise values as the current SLR term.

Data sources:
- Historical shoreline positions: `slpoints_rates.csv.gz`
- Equation-specific Bruun parameters: `output/doc_eq/NZ_*_{hallin|hallout|birk}.csv`
- SLR values: `NZSeaRise_proj_novlm.csv`

This workflow is separate from `step4_bruunrule.py` and does not modify the original pipeline.

## 1. Set Up Environment and Imports

In [ ]:
from pathlib import Path
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output" / "validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CUTOFF = pd.Timestamp("2000-12-31")
VALIDATION_TARGET = pd.Timestamp("2024-12-31")
TRAIN_FRACTION = 0.7

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TRAIN_CUTOFF:", TRAIN_CUTOFF.date())
print("VALIDATION_TARGET:", VALIDATION_TARGET.date())

## 2. Load Historical Shoreline Positions and Build Pre/Post-2000 Targets

In [ ]:
slpoints_path = PROJECT_ROOT / "slpoints_rates.csv.gz"
if not slpoints_path.exists():
    raise FileNotFoundError(f"Missing required file: {slpoints_path}")

sl = pd.read_csv(slpoints_path, low_memory=False)

required_cols = ["Unique_ID", "Date", "IntersectX", "IntersectY"]
missing = [c for c in required_cols if c not in sl.columns]
if missing:
    raise KeyError(f"Missing required columns in slpoints file: {missing}")

sl = sl.copy()
sl["Unique_ID"] = pd.to_numeric(sl["Unique_ID"], errors="coerce").astype("Int64")
sl["Date_dt"] = pd.to_datetime(sl["Date"], errors="coerce", dayfirst=True)
sl["IntersectX"] = pd.to_numeric(sl["IntersectX"], errors="coerce")
sl["IntersectY"] = pd.to_numeric(sl["IntersectY"], errors="coerce")
sl = sl.dropna(subset=["Unique_ID", "Date_dt", "IntersectX", "IntersectY"]).copy()

# Keep one point per transect-date to avoid duplicate-date ambiguity
sl = sl.sort_values(["Unique_ID", "Date_dt"]).drop_duplicates(subset=["Unique_ID", "Date_dt"], keep="last")

first_pts = sl.sort_values("Date_dt").groupby("Unique_ID", as_index=False).first()
pre2000_pts = (
    sl[sl["Date_dt"] <= TRAIN_CUTOFF]
    .sort_values("Date_dt")
    .groupby("Unique_ID", as_index=False)
    .last()
)
latest_to_2024_pts = (
    sl[sl["Date_dt"] <= VALIDATION_TARGET]
    .sort_values("Date_dt")
    .groupby("Unique_ID", as_index=False)
    .last()
)

obs = (
    first_pts[["Unique_ID", "Date_dt", "IntersectX", "IntersectY"]]
    .rename(columns={"Date_dt": "first_date", "IntersectX": "first_x", "IntersectY": "first_y"})
    .merge(
        pre2000_pts[["Unique_ID", "Date_dt", "IntersectX", "IntersectY"]].rename(
            columns={"Date_dt": "pre2000_date", "IntersectX": "pre2000_x", "IntersectY": "pre2000_y"}
        ),
        on="Unique_ID",
        how="inner",
    )
    .merge(
        latest_to_2024_pts[["Unique_ID", "Date_dt", "IntersectX", "IntersectY"]].rename(
            columns={"Date_dt": "latest_date", "IntersectX": "latest_x", "IntersectY": "latest_y"}
        ),
        on="Unique_ID",
        how="inner",
    )
)

# Use a stable landward positive axis from first -> latest(<=2024)
obs["ux_obs"] = obs["latest_x"] - obs["first_x"]
obs["uy_obs"] = obs["latest_y"] - obs["first_y"]
obs["u_norm"] = np.hypot(obs["ux_obs"], obs["uy_obs"])
obs = obs[obs["u_norm"] > 0].copy()
obs["ux_obs"] = obs["ux_obs"] / obs["u_norm"]
obs["uy_obs"] = obs["uy_obs"] / obs["u_norm"]

# Signed projected distances along the transect axis
obs["obs_train_m"] = (obs["pre2000_x"] - obs["first_x"]) * obs["ux_obs"] + (obs["pre2000_y"] - obs["first_y"]) * obs["uy_obs"]
obs["obs_valid_m"] = (obs["latest_x"] - obs["pre2000_x"]) * obs["ux_obs"] + (obs["latest_y"] - obs["pre2000_y"]) * obs["uy_obs"]
obs["obs_total_m"] = (obs["latest_x"] - obs["first_x"]) * obs["ux_obs"] + (obs["latest_y"] - obs["first_y"]) * obs["uy_obs"]

obs["train_years"] = (obs["pre2000_date"] - obs["first_date"]).dt.days / 365.25
obs["valid_years"] = (obs["latest_date"] - obs["pre2000_date"]).dt.days / 365.25

obs = obs[(obs["train_years"] > 0) & (obs["valid_years"] >= 0)].copy()

print("Historical shoreline rows:", len(sl))
print("Transects with full first/pre2000/latest windows:", len(obs))
obs[["Unique_ID", "first_date", "pre2000_date", "latest_date", "obs_train_m", "obs_valid_m"]].head()

## 3. Load Equation Parameters from doc_eq and Current SLR (2020 Medium Confidence)

In [ ]:
# Load equation-specific Bruun parameters from output/doc_eq
# Files contain hallin/hallout/birk variants; *_toe.csv is excluded here.
doc_eq_dir = PROJECT_ROOT / "output" / "doc_eq"
if not doc_eq_dir.exists():
    raise FileNotFoundError(f"Missing directory: {doc_eq_dir}")

eq_suffixes = ["hallin", "hallout", "birk"]

def pick_doc_eq_file(eq):
    candidates = sorted(glob.glob(str(doc_eq_dir / f"NZ_*_{eq}.csv")))
    # Exclude toe variants and keep one representative file (parameters are geometry-driven and expected invariant).
    candidates = [c for c in candidates if not c.lower().endswith("_toe.csv")]
    return candidates[0] if candidates else None

frames = []
for eq in eq_suffixes:
    fp = pick_doc_eq_file(eq)
    if fp is None:
        print(f"WARNING: no doc_eq CSV found for equation={eq}")
        continue

    cols = ["Unique_ID", "Site ID", "R_bruun", "L", "B", "CD", "ux1", "uy1", "point_X", "point_Y"]
    df = pd.read_csv(fp, usecols=cols)
    df["Unique_ID"] = pd.to_numeric(df["Unique_ID"], errors="coerce").astype("Int64")
    df["Site ID"] = pd.to_numeric(df["Site ID"], errors="coerce").astype("Int64")
    for c in ["R_bruun", "L", "B", "CD", "ux1", "uy1", "point_X", "point_Y"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["Unique_ID", "Site ID", "R_bruun"]).copy()
    df["equation"] = eq
    df["source_file"] = Path(fp).name
    df = df.drop_duplicates(subset=["Unique_ID"], keep="first")
    frames.append(df)

if not frames:
    raise ValueError("No usable equation files found in output/doc_eq.")

bruun = pd.concat(frames, ignore_index=True)

# 2020 medium-confidence SLR from NZSeaRise as the 'current' SLR term.
# Note: NZSeaRise file in this workspace starts at 2020 (no 2000/2024 rows).
nzslr_path = PROJECT_ROOT / "NZSeaRise_proj_novlm.csv"
if not nzslr_path.exists():
    raise FileNotFoundError(f"Missing NZSeaRise file: {nzslr_path}")

nz = pd.read_csv(nzslr_path)
req_slr_cols = ["Confidence", "site", "year", "0.5"]
missing_slr = [c for c in req_slr_cols if c not in nz.columns]
if missing_slr:
    raise KeyError(f"Missing NZSeaRise columns: {missing_slr}")

nz["site"] = pd.to_numeric(nz["site"], errors="coerce").astype("Int64")
nz["year"] = pd.to_numeric(nz["year"], errors="coerce")
nz["0.5"] = pd.to_numeric(nz["0.5"], errors="coerce")

slr2020 = (
    nz[(nz["Confidence"] == "medium_confidence") & (nz["year"] == 2020)]
    .groupby("site", as_index=False)["0.5"]
    .median()
    .rename(columns={"site": "Site ID", "0.5": "slr_2020_m"})
)

bruun = bruun.merge(slr2020, on="Site ID", how="left")
found_eqs = sorted(bruun["equation"].dropna().unique().tolist())
print("Bruun records:", len(bruun))
print("Equations found:", found_eqs)
print("SLR-2020 matched rows:", int(bruun["slr_2020_m"].notna().sum()), "/", len(bruun))
if len(found_eqs) < 3:
    print("WARNING: fewer than 3 equations loaded from output/doc_eq.")
bruun.head()

## 4. Pseudo-Forecast from 2000 to 2024 Using Pre-2000 Historic Trend + Bruun SLR Term

In [ ]:
# Merge observed pre/post-2000 distances with equation parameters
data = bruun.merge(obs, on="Unique_ID", how="inner")

data = data.dropna(subset=["obs_train_m", "obs_valid_m", "train_years", "valid_years", "R_bruun", "slr_2020_m"]).copy()

# Historic trend calibrated from observations available before 2000
data["historic_rate_m_per_yr"] = data["obs_train_m"] / data["train_years"]

# Pseudo-forecast from ~2000 to latest<=2024:
# predicted = historic_trend_component + SLR_component
# where SLR_component = (retreat per meter SLR) * (current absolute SLR)
# and R_bruun here is the per-meter Bruun sensitivity term from doc_eq outputs.
data["hist_component_m"] = data["historic_rate_m_per_yr"] * data["valid_years"]
data["slr_component_m"] = data["R_bruun"] * data["slr_2020_m"]
data["pred_valid_m"] = data["hist_component_m"] + data["slr_component_m"]

# Position projection along observed transect axis
data["pred_latest_x"] = data["pre2000_x"] + data["ux_obs"] * data["pred_valid_m"]
data["pred_latest_y"] = data["pre2000_y"] + data["uy_obs"] * data["pred_valid_m"]
data["xy_err_m"] = np.hypot(data["pred_latest_x"] - data["latest_x"], data["pred_latest_y"] - data["latest_y"])

data["resid_valid_m"] = data["pred_valid_m"] - data["obs_valid_m"]

print("Rows after merge/filters:", len(data))
print("Unique transects:", data["Unique_ID"].nunique())
print("Unique equations:", sorted(data["equation"].unique().tolist()))
data[["equation", "Unique_ID", "obs_train_m", "obs_valid_m", "hist_component_m", "slr_component_m", "pred_valid_m", "xy_err_m"]].head()

## 5. Score Equation Accuracy Against Observed 2000-2024 Change

In [ ]:
def metric_summary(df):
    d = df[["obs_valid_m", "pred_valid_m", "xy_err_m"]].dropna().copy()
    if d.empty:
        return pd.Series({"n": 0, "bias_m": np.nan, "mae_m": np.nan, "rmse_m": np.nan, "corr": np.nan, "xy_mae_m": np.nan, "xy_rmse_m": np.nan})

    obs_v = d["obs_valid_m"].to_numpy(dtype=float)
    pred_v = d["pred_valid_m"].to_numpy(dtype=float)
    res = pred_v - obs_v

    corr = np.nan
    if len(d) > 2 and np.nanstd(obs_v) > 0 and np.nanstd(pred_v) > 0:
        corr = np.corrcoef(obs_v, pred_v)[0, 1]

    return pd.Series(
        {
            "n": len(d),
            "bias_m": float(np.nanmean(res)),
            "mae_m": float(np.nanmean(np.abs(res))),
            "rmse_m": float(np.sqrt(np.nanmean(res**2))),
            "corr": corr,
            "xy_mae_m": float(np.nanmean(d["xy_err_m"])),
            "xy_rmse_m": float(np.sqrt(np.nanmean(d["xy_err_m"] ** 2))),
        }
    )

metrics_by_eq = data.groupby("equation", dropna=False).apply(metric_summary).reset_index()
metrics_by_eq = metrics_by_eq.sort_values(["rmse_m", "mae_m", "xy_rmse_m"], ascending=True).reset_index(drop=True)

display(metrics_by_eq)

if len(metrics_by_eq) > 0:
    best_eq = metrics_by_eq.iloc[0]["equation"]
    print("Best-performing equation:", best_eq)
else:
    best_eq = None

## 6. Visual Diagnostics for the Best Equation

In [ ]:
if best_eq is None:
    raise ValueError("No equation metrics available to plot.")

plot_df = data[data["equation"] == best_eq].copy()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

sns.scatterplot(data=plot_df, x="obs_valid_m", y="pred_valid_m", ax=axes[0], s=22, alpha=0.65)
lims = [
    np.nanmin([plot_df["obs_valid_m"].min(), plot_df["pred_valid_m"].min()]),
    np.nanmax([plot_df["obs_valid_m"].max(), plot_df["pred_valid_m"].max()]),
]
axes[0].plot(lims, lims, "k--", linewidth=1)
axes[0].set_title(f"{best_eq}: Observed vs Predicted (2000->2024)")
axes[0].set_xlabel("Observed change (m)")
axes[0].set_ylabel("Predicted change (m)")

sns.histplot(plot_df["resid_valid_m"], bins=35, kde=True, ax=axes[1], color="#2C7FB8")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("pred - obs (m)")

comp = plot_df[["hist_component_m", "slr_component_m"]].melt(var_name="component", value_name="distance_m")
sns.boxplot(data=comp, x="component", y="distance_m", ax=axes[2])
axes[2].set_title("Projection Components")
axes[2].set_xlabel("")
axes[2].set_ylabel("Distance (m)")

plt.tight_layout()
plt.show()

## 7. Compare All Equations Side by Side

In [ ]:
if len(metrics_by_eq) == 0:
    print("No equation rows available for comparison.")
else:
    plt.figure(figsize=(8.5, 4.4))
    sns.barplot(data=metrics_by_eq, x="equation", y="rmse_m", palette="deep")
    plt.title("Validation RMSE by Equation (2000->2024)")
    plt.xlabel("Equation")
    plt.ylabel("RMSE (m)")
    plt.tight_layout()
    plt.show()

    # Optional: show equation-level contribution means
    contrib = (
        data.groupby("equation", dropna=False)[["hist_component_m", "slr_component_m"]]
        .mean()
        .reset_index()
    )
    display(contrib)

## 8. Export Validation Outputs

In [ ]:
obs_out = OUTPUT_DIR / "nzccd_observed_pre2000_post2000_targets.csv"
params_out = OUTPUT_DIR / "bruun_parameters_with_slr2020.csv"
rows_out = OUTPUT_DIR / "bruun_validation_rows_pre2000_to_2024.csv"
metrics_out = OUTPUT_DIR / "bruun_validation_metrics_by_equation.csv"

obs.to_csv(obs_out, index=False)
bruun.to_csv(params_out, index=False)
data.to_csv(rows_out, index=False)
metrics_by_eq.to_csv(metrics_out, index=False)

# Save one summary scatter for the winning equation
fig_path = None
if best_eq is not None:
    p = data[data["equation"] == best_eq].copy()
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.scatterplot(data=p, x="obs_valid_m", y="pred_valid_m", ax=ax, s=22, alpha=0.65)
    lims = [
        np.nanmin([p["obs_valid_m"].min(), p["pred_valid_m"].min()]),
        np.nanmax([p["obs_valid_m"].max(), p["pred_valid_m"].max()]),
    ]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_title(f"Best Equation: {best_eq}")
    ax.set_xlabel("Observed 2000->2024 change (m)")
    ax.set_ylabel("Predicted 2000->2024 change (m)")
    plt.tight_layout()
    fig_path = OUTPUT_DIR / "best_equation_obs_vs_pred_2000_2024.png"
    fig.savefig(fig_path, dpi=220)
    plt.close(fig)

print("Export complete:")
print("-", obs_out)
print("-", params_out)
print("-", rows_out)
print("-", metrics_out)
if fig_path is not None:
    print("-", fig_path)